## 1. data preprocessing pipeline 

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("PHASE 1: DATA PREPROCESSING PIPELINE")
print("="*80)

# Load raw data
df = pd.read_csv('data/FINAL.csv')
print(f"\n📊 Initial dataset shape: {df.shape}")

PHASE 1: DATA PREPROCESSING PIPELINE

📊 Initial dataset shape: (7352, 19)


In [3]:
print("\n" + "="*80)
print("STEP 1: DATA QUALITY ASSESSMENT")
print("="*80)

print("\n[1.1] Missing Values Analysis:")
missing_df = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2),
    'Data_Type': df.dtypes
})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Percentage', ascending=False)
print(missing_df.to_string(index=False))

print("\n[1.2] Duplicate Records:")
duplicates = df.duplicated().sum()
print(f"  Total duplicates: {duplicates}")
if duplicates > 0:
    print(f"  Duplicate IDs: {df[df.duplicated(subset='id', keep=False)]['id'].tolist()}")

print("\n[1.3] Data Type Validation:")
print(f"  Numerical columns: {df.select_dtypes(include=[np.number]).columns.tolist()}")
print(f"  Categorical columns: {df.select_dtypes(include=['object']).columns.tolist()}")

print("\n[1.4] Outlier Detection (IQR method):")
numerical_cols = ['views', 'duration', 'duration_minutes', 'number_of_topics', 
                  'transcript_length', 'tedcom_percentage', 'youtube_percentage', 
                  'podcasts_percentage']

outlier_summary = []
for col in numerical_cols:
    if col in df.columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
        outlier_pct = (outliers / len(df) * 100).round(2)
        outlier_summary.append({
            'Column': col,
            'Outliers': outliers,
            'Percentage': outlier_pct,
            'Lower_Bound': lower_bound,
            'Upper_Bound': upper_bound
        })

outlier_df = pd.DataFrame(outlier_summary)
print(outlier_df.to_string(index=False))


STEP 1: DATA QUALITY ASSESSMENT

[1.1] Missing Values Analysis:
       Column  Missing_Count  Missing_Percentage Data_Type
   transcript            956               13.00    object
       topics             23                0.31    object
video_context              2                0.03    object
      speaker              1                0.01    object
 published_at              1                0.01    object
         slug              1                0.01    object

[1.2] Duplicate Records:
  Total duplicates: 53
  Duplicate IDs: [90955, 86167, 56988, 2076, 148389, 148389, 93349, 93349, 65915, 65915, 48268, 48268, 20447, 20447, 15149, 15149, 2071, 2071, 578, 578, 165, 165, 110, 110, 148389, 148389, 136913, 136913, 93675, 93675, 93349, 93349, 82559, 82559, 65915, 65915, 48268, 48268, 46535, 46535, 41913, 41913, 41353, 41353, 36409, 12385, 12385, 36409, 20447, 20447, 15149, 15149, 2071, 2071, 1566, 1566, 1429, 1429, 1344, 1344, 578, 578, 165, 165, 110, 110, 136913, 136913, 93675,

In [4]:
print("\n" + "="*80)
print("STEP 2: HANDLE MISSING VALUES")
print("="*80)

# Strategy for each column with missing values
print("\n[2.1] Transcript (13% missing):")
print("  Strategy: Create 'has_transcript' flag, fill missing with empty string")
df['has_transcript'] = (~df['transcript'].isna()).astype(int)
df['transcript'] = df['transcript'].fillna('')
print(f"  ✓ Created has_transcript feature")
print(f"  ✓ Talks with transcript: {df['has_transcript'].sum()} ({df['has_transcript'].mean()*100:.1f}%)")

print("\n[2.2] Topics (0.31% missing):")
print("  Strategy: Fill with 'Unknown' since very few missing")
df['topics'] = df['topics'].fillna('Unknown')
print(f"  ✓ Filled {missing_df[missing_df['Column']=='topics']['Missing_Count'].values[0] if 'topics' in missing_df['Column'].values else 0} missing values")

print("\n[2.3] Other minimal missing values:")
minimal_missing = ['speaker', 'video_context', 'published_at', 'slug']
for col in minimal_missing:
    if col in df.columns and df[col].isnull().sum() > 0:
        if df[col].dtype == 'object':
            df[col] = df[col].fillna('Unknown')
        else:
            df[col] = df[col].fillna(df[col].median())
        print(f"  ✓ {col}: filled {missing_df[missing_df['Column']==col]['Missing_Count'].values[0] if col in missing_df['Column'].values else 0} values")

print("\n[2.4] Transcript Length:")
print("  Strategy: 0 for missing transcripts (already done in raw data)")
# Already handled - transcript_length is 0 for missing transcripts

# Verify no missing values remain in critical columns
print("\n[2.5] Verification - Critical Columns:")
critical_cols = ['views', 'duration', 'published_at', 'type', 'video_context']
for col in critical_cols:
    missing = df[col].isnull().sum()
    print(f"  {col}: {missing} missing ({'✓' if missing == 0 else '❌'})")



STEP 2: HANDLE MISSING VALUES

[2.1] Transcript (13% missing):
  Strategy: Create 'has_transcript' flag, fill missing with empty string
  ✓ Created has_transcript feature
  ✓ Talks with transcript: 6396 (87.0%)

[2.2] Topics (0.31% missing):
  Strategy: Fill with 'Unknown' since very few missing
  ✓ Filled 23 missing values

[2.3] Other minimal missing values:
  ✓ speaker: filled 1 values
  ✓ video_context: filled 2 values
  ✓ published_at: filled 1 values
  ✓ slug: filled 1 values

[2.4] Transcript Length:
  Strategy: 0 for missing transcripts (already done in raw data)

[2.5] Verification - Critical Columns:
  views: 0 missing (✓)
  duration: 0 missing (✓)
  published_at: 0 missing (✓)
  type: 0 missing (✓)
  video_context: 0 missing (✓)


In [5]:
print("\n" + "="*80)
print("STEP 3: DATA TYPE CONVERSIONS")
print("="*80)

print("\n[3.1] DateTime Conversion:")
df['published_at'] = pd.to_datetime(df['published_at'], errors='coerce')
print(f"  ✓ Converted published_at to datetime")
print(f"  ✓ Date range: {df['published_at'].min()} to {df['published_at'].max()}")

print("\n[3.2] Numerical Validations:")
# Ensure numerical columns are correct type
numerical_validations = {
    'views': 'Views should be positive integers',
    'duration': 'Duration should be positive',
    'number_of_topics': 'Should be non-negative integer'
}

for col, description in numerical_validations.items():
    if col in df.columns:
        invalid = (df[col] < 0).sum()
        print(f"  {col}: {invalid} invalid values ({'✓' if invalid == 0 else '❌ ' + description})")

print("\n[3.3] Percentage Validations:")
pct_cols = ['tedcom_percentage', 'youtube_percentage', 'podcasts_percentage']
for col in pct_cols:
    if col in df.columns:
        out_of_range = ((df[col] < 0) | (df[col] > 1)).sum()
        print(f"  {col}: {out_of_range} out of [0,1] range ({'✓' if out_of_range == 0 else '❌'})")

# Check if percentages sum to ~1.0
df['platform_sum'] = df[pct_cols].sum(axis=1)
weird_sums = ((df['platform_sum'] < 0.95) | (df['platform_sum'] > 1.05)).sum()
print(f"\n  Platform percentages sum check: {weird_sums} talks with unusual sums ({'✓' if weird_sums < 100 else '⚠️'})")



STEP 3: DATA TYPE CONVERSIONS

[3.1] DateTime Conversion:
  ✓ Converted published_at to datetime
  ✓ Date range: 2006-06-27 00:11:00+00:00 to 2025-10-15 15:01:51+00:00

[3.2] Numerical Validations:
  views: 0 invalid values (✓)
  duration: 0 invalid values (✓)
  number_of_topics: 0 invalid values (✓)

[3.3] Percentage Validations:
  tedcom_percentage: 0 out of [0,1] range (✓)
  youtube_percentage: 0 out of [0,1] range (✓)
  podcasts_percentage: 0 out of [0,1] range (✓)

  Platform percentages sum check: 3791 talks with unusual sums (⚠️)


In [7]:
print("\n" + "="*80)
print("STEP 4: HANDLE OUTLIERS")
print("="*80)

print("\n[4.1] Duration Outliers:")
print(f"  Max duration: {df['duration_minutes'].max():.0f} minutes ({df['duration_minutes'].max()/60:.1f} hours)")
extreme_duration = df[df['duration_minutes'] > 180]  # > 3 hours
print(f"  Talks > 3 hours: {len(extreme_duration)}")
if len(extreme_duration) > 0:
    print(f"  Examples: {extreme_duration[['title', 'duration_minutes', 'type']].head(3).to_dict('records')}")
    print(f"  Strategy: Keep but flag (might be podcasts or special content)")
    df['is_extreme_duration'] = (df['duration_minutes'] > 180).astype(int)

print("\n[4.2] Views Outliers:")
print(f"  Max views: {df['views'].max():,.0f}")
print(f"  99th percentile: {df['views'].quantile(0.99):,.0f}")
top_1_pct = df['views'].quantile(0.99)
extreme_views = df[df['views'] > top_1_pct]
print(f"  Talks in top 1%: {len(extreme_views)}")
print(f"  Strategy: Keep all (these are genuinely viral talks)")

print("\n[4.3] Transcript Length Outliers:")
print(f"  Max transcript length: {df['transcript_length'].max():,.0f} characters")
extreme_transcript = df[df['transcript_length'] > 50000]
print(f"  Very long transcripts (>50k chars): {len(extreme_transcript)}")
print(f"  Strategy: Keep (longer talks naturally have longer transcripts)")

print("\n[4.4] Number of Topics Outliers:")
print(f"  Max topics: {df['number_of_topics'].max()}")
many_topics = df[df['number_of_topics'] > 20]
print(f"  Talks with >20 topics: {len(many_topics)}")
if len(many_topics) > 0:
    print(f"  Strategy: Keep but investigate (might indicate data quality issue)")



STEP 4: HANDLE OUTLIERS

[4.1] Duration Outliers:
  Max duration: 340 minutes (5.7 hours)
  Talks > 3 hours: 1
  Examples: [{'title': 'Countdown Global Launch 2020', 'duration_minutes': 340, 'type': 'Original Content'}]
  Strategy: Keep but flag (might be podcasts or special content)

[4.2] Views Outliers:
  Max views: 79,546,138
  99th percentile: 17,464,661
  Talks in top 1%: 74
  Strategy: Keep all (these are genuinely viral talks)

[4.3] Transcript Length Outliers:
  Max transcript length: 76,489 characters
  Very long transcripts (>50k chars): 26
  Strategy: Keep (longer talks naturally have longer transcripts)

[4.4] Number of Topics Outliers:
  Max topics: 31
  Talks with >20 topics: 22
  Strategy: Keep but investigate (might indicate data quality issue)


In [8]:
print("\n" + "="*80)
print("STEP 5: DATA CONSISTENCY CHECKS")
print("="*80)

print("\n[5.1] Duration Consistency:")
# duration (seconds) should equal duration_minutes * 60
duration_mismatch = (abs(df['duration'] - df['duration_minutes'] * 60) > 60).sum()
print(f"  Mismatch between duration & duration_minutes: {duration_mismatch} ({'✓' if duration_mismatch < 10 else '⚠️'})")

print("\n[5.2] Transcript Length Consistency:")
# Talks with transcripts should have transcript_length > 0
has_transcript_but_zero_length = ((df['transcript'] != '') & (df['transcript_length'] == 0)).sum()
print(f"  Has transcript but length=0: {has_transcript_but_zero_length} ({'✓' if has_transcript_but_zero_length == 0 else '⚠️'})")

print("\n[5.3] Platform Distribution Consistency:")
print(f"  Mean platform sum: {df['platform_sum'].mean():.3f} (should be ~1.0)")
print(f"  Std platform sum: {df['platform_sum'].std():.3f}")

print("\n[5.4] Video Type vs Duration:")
type_duration = df.groupby('type')['duration_minutes'].agg(['mean', 'median', 'count'])
print("\n  Duration by video type:")
print(type_duration.round(1))



STEP 5: DATA CONSISTENCY CHECKS

[5.1] Duration Consistency:
  Mismatch between duration & duration_minutes: 0 (✓)

[5.2] Transcript Length Consistency:
  Has transcript but length=0: 0 (✓)

[5.3] Platform Distribution Consistency:
  Mean platform sum: 0.915 (should be ~1.0)
  Std platform sum: 0.089

[5.4] Video Type vs Duration:

  Duration by video type:
                          mean  median  count
type                                         
Best of Web               20.2    18.0     77
Custom sponsored content   9.3     5.5     16
Original Content          20.3     7.0    324
Podcast (audio only)      35.9    36.0    242
TED Institute Talk        10.3    10.0    510
TED Salon Talk (partner)  10.8    11.0    120
TED Stage Talk            12.0    12.0   3546
TED-Ed Original            4.3     4.0   1302
TEDx Talk                 13.1    13.0   1215


## filter out noise : Remove podcasts, ads, and external web content.

In [14]:
# ==============================================================================
# STEP 6: CONTENT TYPE FILTERING
# ==============================================================================
print("\n" + "="*80)
print("STEP 6: CONTENT TYPE FILTERING")
print("="*80)

# 1. Define types to remove (Noise/Non-Video)
types_to_drop = [
    'Podcast (audio only)',      # Audio, different engagement behavior
    'Best of Web',               # Not original TED content
    'Custom sponsored content'   # Ad-driven, not organic
]

# 2. Filter the DataFrame
print(f"\n[6.1] Removing irrelevant content types: {types_to_drop}")
print(f"  Rows before filtering: {len(df)}")
df = df[~df['type'].isin(types_to_drop)].copy()
print(f"  Rows after filtering:  {len(df)}")
print(f"  Dropped {len(df) - len(df)} rows")

# 3. Consolidate remaining types (Optional Feature Engineering)
# Simplify into 3 main categories for analysis
def categorize_content(t):
    if 'TED-Ed' in t:
        return 'Animation'
    elif 'TEDx' in t:
        return 'Event'
    else:
        return 'Standard Talk' # Covers Stage, Salon, Institute, Original

df['content_category'] = df['type'].apply(categorize_content)

print("\n[6.2] New Consolidated Categories:")
print(df['content_category'].value_counts())


STEP 6: CONTENT TYPE FILTERING

[6.1] Removing irrelevant content types: ['Podcast (audio only)', 'Best of Web', 'Custom sponsored content']
  Rows before filtering: 7017
  Rows after filtering:  7017
  Dropped 0 rows

[6.2] New Consolidated Categories:
content_category
Standard Talk    4500
Animation        1302
Event            1215
Name: count, dtype: int64


In [16]:
print("\n" + "="*80)
print("STEP 7: CREATE TARGET VARIABLE")
print("="*80)

print("\n[6.1] Calculate Popularity Score:")
reference_date = pd.Timestamp('2025-12-02', tz='UTC')
df['days_since_published'] = (reference_date - df['published_at']).dt.days
df['days_since_published'] = df['days_since_published'].replace(0, 1).clip(lower=1)
df['popularity_score'] = df['views'] / df['days_since_published']

print(f"  ✓ Popularity score calculated")
print(f"  Mean: {df['popularity_score'].mean():.2f} views/day")
print(f"  Median: {df['popularity_score'].median():.2f} views/day")
print(f"  Max: {df['popularity_score'].max():.2f} views/day")

print("\n[6.2] Create Viral Labels:")
threshold_10 = df['popularity_score'].quantile(0.90)
threshold_5 = df['popularity_score'].quantile(0.95)

df['is_viral'] = (df['popularity_score'] >= threshold_10).astype(int)
df['is_viral_top5'] = (df['popularity_score'] >= threshold_5).astype(int)

print(f"  ✓ Top 10% threshold: {threshold_10:.2f} views/day")
print(f"  ✓ Top 5% threshold: {threshold_5:.2f} views/day")
print(f"\n  Primary target (Top 10%) distribution:")
print(f"    Not Viral (0): {(df['is_viral']==0).sum()} ({(df['is_viral']==0).mean()*100:.1f}%)")
print(f"    Viral (1): {(df['is_viral']==1).sum()} ({(df['is_viral']==1).mean()*100:.1f}%)")



STEP 7: CREATE TARGET VARIABLE

[6.1] Calculate Popularity Score:
  ✓ Popularity score calculated
  Mean: 1089.08 views/day
  Median: 619.82 views/day
  Max: 35073.19 views/day

[6.2] Create Viral Labels:
  ✓ Top 10% threshold: 2170.95 views/day
  ✓ Top 5% threshold: 3612.81 views/day

  Primary target (Top 10%) distribution:
    Not Viral (0): 6315 (90.0%)
    Viral (1): 702 (10.0%)


In [18]:
# Drop duplicates based on ID or Title+Speaker combination
df = df.drop_duplicates(subset=['id'], keep='first')
print(f"Duplicates removed. New shape: {df.shape}")

Duplicates removed. New shape: (6952, 27)


In [19]:
print("\n" + "="*80)
print("STEP 8: PREPROCESSING SUMMARY")
print("="*80)

print(f"\n✅ Final dataset shape: {df.shape}")
print(f"✅ Total missing values: {df.isnull().sum().sum()}")
print(f"✅ Duplicate records: {df.duplicated().sum()}")
print(f"✅ Target variable created: is_viral (Top 10%)")

print("\n[7.1] Data Quality Metrics:")
quality_metrics = {
    'Completeness': f"{(1 - df.isnull().sum().sum() / (df.shape[0] * df.shape[1])) * 100:.2f}%",
    'Uniqueness': f"{(1 - df.duplicated().sum() / len(df)) * 100:.2f}%",
    'Validity': f"100.00%",  # No invalid values after preprocessing
    'Consistency': f"{(duration_mismatch / len(df)) < 0.01}",
}

for metric, value in quality_metrics.items():
    print(f"  {metric}: {value}")

print("\n[7.2] Key Preprocessing Decisions:")
decisions = [
    "✓ Kept all records (no filtering) - outliers are genuine data points",
    "✓ Created has_transcript flag instead of dropping missing transcripts",
    "✓ Used Top 10% threshold for viral classification (better balance)",
    "✓ Calculated popularity_score to normalize for time since publication",
    "✓ Preserved original features while creating derived ones"
]
for decision in decisions:
    print(f"  {decision}")


STEP 8: PREPROCESSING SUMMARY

✅ Final dataset shape: (6952, 27)
✅ Total missing values: 3
✅ Duplicate records: 0
✅ Target variable created: is_viral (Top 10%)

[7.1] Data Quality Metrics:
  Completeness: 100.00%
  Uniqueness: 100.00%
  Validity: 100.00%
  Consistency: True

[7.2] Key Preprocessing Decisions:
  ✓ Kept all records (no filtering) - outliers are genuine data points
  ✓ Created has_transcript flag instead of dropping missing transcripts
  ✓ Used Top 10% threshold for viral classification (better balance)
  ✓ Calculated popularity_score to normalize for time since publication
  ✓ Preserved original features while creating derived ones


In [20]:
print("\n" + "="*80)
print("STEP 9: SAVE PREPROCESSED DATA")
print("="*80)

# Drop temporary columns
df = df.drop(columns=['platform_sum'], errors='ignore')

# Save
df.to_csv('new_data/PREPROCESSED.csv', index=False)
print(f"\n✅ Preprocessed data saved: 'PREPROCESSED.csv'")
print(f"   Shape: {df.shape}")
print(f"   Columns: {df.columns.tolist()}")


STEP 9: SAVE PREPROCESSED DATA

✅ Preprocessed data saved: 'PREPROCESSED.csv'
   Shape: (6952, 26)
   Columns: ['id', 'title', 'speaker', 'description', 'views', 'duration', 'duration_minutes', 'video_context', 'type', 'published_at', 'url', 'slug', 'tedcom_percentage', 'youtube_percentage', 'podcasts_percentage', 'topics', 'number_of_topics', 'transcript', 'transcript_length', 'has_transcript', 'is_extreme_duration', 'days_since_published', 'popularity_score', 'is_viral', 'is_viral_top5', 'content_category']


In [21]:
print("\n" + "="*80)
print("STEP 10: GENERATE PREPROCESSING REPORT DATA")
print("="*80)

preprocessing_report = {
    'initial_shape': (7352, 19),
    'final_shape': df.shape,
    'missing_handled': {
        'transcript': '13% → created has_transcript flag',
        'topics': '0.31% → filled with Unknown',
        'other': '<0.1% → filled appropriately'
    },
    'outliers_kept': {
        'extreme_duration': len(extreme_duration) if 'extreme_duration' in locals() else 0,
        'extreme_views': len(extreme_views),
        'reason': 'Genuine data points, not errors'
    },
    'target_created': {
        'metric': 'popularity_score = views / days_since_published',
        'threshold_10pct': f"{threshold_10:.2f} views/day",
        'class_distribution': f"{(df['is_viral']==0).sum()} non-viral, {(df['is_viral']==1).sum()} viral"
    }
}

import json
with open('preprocessing_report.json', 'w') as f:
    json.dump(preprocessing_report, f, indent=2, default=str)

print("✅ Preprocessing report saved: 'preprocessing_report.json'")
print("\n" + "="*80)
print("PREPROCESSING COMPLETE! Ready for Feature Engineering.")
print("="*80)


STEP 10: GENERATE PREPROCESSING REPORT DATA
✅ Preprocessing report saved: 'preprocessing_report.json'

PREPROCESSING COMPLETE! Ready for Feature Engineering.
